In [3]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/final.csv")


In [4]:
print(df.columns)

Index(['time_id', 'seconds_in_bucket', '0', '1', '2', '3', '4', '5', '6', '7',
       ...
       '115', '116', '118', '119', '120', '122', '123', '124', '125', '126'],
      dtype='object', length=114)


In [5]:
print(df.head())

   time_id  seconds_in_bucket           0           1           2           3  \
0        5                  0  193.649150  152.533206  123.541146  226.045064   
1        5                  1  193.651874  152.570918  123.524616  226.035539   
2        5                  2  193.651874  152.532436  123.524567  225.995551   
3        5                  3  193.651874  152.628598  123.539480  225.986970   
4        5                  4  193.651874  152.583596  123.524567  225.986970   

            4           5           6           7  ...        115         116  \
0  620.055416  740.619940  370.858389  246.111936  ...  88.115209  245.587060   
1  620.049963  740.623669  370.584903  246.074052  ...  88.123758  245.695881   
2  620.049963  740.623669  370.584903  246.097057  ...  88.116026  245.698838   
3  620.057978  740.624582  370.584903  246.097057  ...  88.141892  245.698838   
4  620.049963  740.626447  370.584903  246.097057  ...  88.141892  245.702744   

          118        119  

In [ ]:
"""
lstm_cluster_pipeline.py — Multi-Stock Cluster LSTM Volatility
================================================================

Architecture (60 × 12 input):
  • Input         : (B, K, 60, 12) — 60 time steps, 12 feature channels
  • LSTM          : 3 stacked layers × 256 hidden units, dropout p=0.2
  • Output        : FC linear → single scalar per cluster
  • Scalar branch : HAR-RV MLP correction per cluster
  • Head          : concat(LSTM repr, scalar repr) → linear → (B, K)

Windowing
---------
  Each time_id has exactly 600 rows (seconds_in_bucket).
  Non-overlapping windows of 60 input + 120 target = 180 steps
  yield 3 samples per time_id:
      sample 0 : input [  0: 60), target [ 60:180)
      sample 1 : input [180:240), target [240:360)
      sample 2 : input [360:420), target [420:540)
  Rows 540–599 are unused.

12 Feature Channels
-------------------
  0  return          — clipped log-return
  1  log_wap         — log(WAP / open)
  2  rv_short        — sqrt(Σ ret²,  5-step window)
  3  rv_medium       — sqrt(Σ ret², 20-step window)
  4  rv_long         — sqrt(Σ ret², 50-step window)
  5  bpv             — bipower variation (5-step)
  6  jump            — rv_short² − bpv, floored at 0
  7  signed_rv       — ret × |ret|  (signed quadratic variation)
  8  momentum        — Σ ret over 10-step rolling window
  9  rv_ratio        — rv_short / (rv_long + ε)  (vol-of-vol proxy)
 10  abs_ret_ma      — rolling mean |ret| over 5-step window
 11  ret_zscore      — (ret − μ̂₂₀) / (σ̂₂₀ + ε)  (standardised return)

Evaluation
----------
  Both smearing-corrected and raw (non-smearing) RV metrics are
  reported side-by-side so the correction's contribution is visible.
"""

import warnings
import time as _time

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    from kneed import KneeLocator
    _KNEED_AVAILABLE = True
except ImportError:
    _KNEED_AVAILABLE = False


# ─────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────

INPUT_LEN  = 60        # 60 time steps fed to LSTM
TARGET_LEN = 120       # 120-step forward window for RV target
STRIDE     = INPUT_LEN + TARGET_LEN   # 180 — non-overlapping window size
TOTAL_LEN  = 600       # each time_id has exactly 600 rows
SAMPLES_PER_TID = TOTAL_LEN // STRIDE  # 3 samples per time_id

EPSILON    = 1e-5
RET_CLIP   = 0.05

#  12 feature channels (see module docstring for definitions)
CHANNEL_NAMES = [
    "return",       # 0
    "log_wap",      # 1
    "rv_short",     # 2
    "rv_medium",    # 3
    "rv_long",      # 4
    "bpv",          # 5
    "jump",         # 6
    "signed_rv",    # 7
    "momentum",     # 8
    "rv_ratio",     # 9
    "abs_ret_ma",   # 10
    "ret_zscore",   # 11
]
N_CHANNELS         = len(CHANNEL_NAMES)          # 12
PROFILE_COLS       = ["rv", "mean_ret", "skew", "kurt", "max_dd"]
SCALAR_CHANNEL_IDX = [2, 3, 4, 5, 6]            # rv_short … jump


# ─────────────────────────────────────────────────────────────────────
# LOW-LEVEL FEATURE HELPERS
# ─────────────────────────────────────────────────────────────────────

def _safe_log_return_matrix(mat: np.ndarray) -> np.ndarray:
    """(T, S) WAP → (T, S) clipped log-return matrix."""
    safe = np.where(mat > 0, mat, 1e-6).astype(np.float32)
    ret  = np.zeros_like(safe)
    ret[1:] = np.log(safe[1:] / safe[:-1])
    np.clip(ret, -RET_CLIP, RET_CLIP, out=ret)
    np.nan_to_num(ret, copy=False, nan=0.0, posinf=RET_CLIP, neginf=-RET_CLIP)
    return ret


def _rolling_sum_2d(mat: np.ndarray, w: int) -> np.ndarray:
    """(T, S) → (T, S) rolling sum with window w (vectorised, O(T·S))."""
    T, S = mat.shape
    cs   = np.zeros((T + 1, S), dtype=np.float64)
    np.cumsum(mat, axis=0, out=cs[1:])
    idx  = np.arange(1, T + 1)
    return (cs[idx] - cs[np.maximum(idx - w, 0)]).astype(np.float32)


def _rolling_mean_std_2d(mat: np.ndarray, w: int):
    """Returns (rolling_mean, rolling_std) both (T, S)."""
    rm  = _rolling_sum_2d(mat, w) / w
    sq  = _rolling_sum_2d(mat ** 2, w) / w
    var = np.maximum(sq - rm ** 2, 0.0)
    return rm.astype(np.float32), np.sqrt(var).astype(np.float32)


def robust_fill(mat: np.ndarray) -> np.ndarray:
    """Forward/backward fill NaN / non-positive values column-wise."""
    bad = ~np.isfinite(mat) | (mat <= 0)
    if not bad.any():
        return mat
    out     = mat.copy()
    rows    = out.shape[0]
    row_idx = np.arange(rows)[:, None]
    fwd     = np.where(~bad, row_idx, -1)
    np.maximum.accumulate(fwd, axis=0, out=fwd)
    rev = np.where(~bad, row_idx, rows)
    bwd = rows - 1 - np.maximum.accumulate(
        (rows - 1 - rev)[::-1], axis=0
    )[::-1]
    use     = np.where(fwd >= 0, fwd, bwd)
    all_bad = (use < 0) | (use >= rows)
    use     = np.where(all_bad, 0, use)
    out     = out[use, np.arange(out.shape[1])]
    return np.where(all_bad, 1.0, out)


def compute_features_matrix(wap_mat: np.ndarray,
                             open_prices: np.ndarray) -> np.ndarray:
    """
    (T, S) WAP matrix → (T, S, 12) feature tensor.

    Feature layout matches CHANNEL_NAMES / N_CHANNELS = 12.
    """
    T, S = wap_mat.shape

    # ── Channels 0–1: return, log_wap ────────────────────────────────
    ret      = _safe_log_return_matrix(wap_mat)                     # (T, S)
    safe_wap = np.maximum(wap_mat, 1e-6).astype(np.float32)
    log_wap  = np.log(
        safe_wap / (open_prices[None, :] + EPSILON)
    ).astype(np.float32)

    # ── Channels 2–4: rolling RVs ────────────────────────────────────
    sq_ret    = (ret ** 2).astype(np.float32)
    rv_short  = np.sqrt(_rolling_sum_2d(sq_ret,  5))                # (T, S)
    rv_medium = np.sqrt(_rolling_sum_2d(sq_ret, 20))
    rv_long   = np.sqrt(_rolling_sum_2d(sq_ret, 50))

    # ── Channels 5–6: bipower variation & jump ───────────────────────
    abs_ret   = np.abs(ret)
    product   = np.zeros_like(abs_ret)
    product[1:] = abs_ret[1:] * abs_ret[:-1]
    bpv  = (_rolling_sum_2d(product, 5) * (np.pi / 2)).astype(np.float32)
    bpv[:2] = 0.0
    jump = np.maximum(_rolling_sum_2d(sq_ret, 5) - bpv, 0.0).astype(np.float32)

    # ── Channel 7: signed RV  (direction-aware quadratic variation) ──
    signed_rv = (ret * abs_ret).astype(np.float32)

    # ── Channel 8: momentum  (10-step cumulative return) ─────────────
    momentum = _rolling_sum_2d(ret, 10).astype(np.float32)

    # ── Channel 9: rv_ratio  (vol-of-vol proxy) ──────────────────────
    rv_ratio = (rv_short / (rv_long + EPSILON)).astype(np.float32)

    # ── Channel 10: abs_ret_ma  (mean absolute return, 5-step) ───────
    abs_ret_ma = (_rolling_sum_2d(abs_ret, 5) / 5).astype(np.float32)

    # ── Channel 11: return z-score  (standardised over 20-step window)
    r_mu, r_sigma = _rolling_mean_std_2d(ret, 20)
    ret_zscore    = ((ret - r_mu) / (r_sigma + EPSILON)).astype(np.float32)

    feat = np.stack(
        [ret, log_wap, rv_short, rv_medium, rv_long,
         bpv, jump, signed_rv, momentum, rv_ratio, abs_ret_ma, ret_zscore],
        axis=-1,
    )  # (T, S, 12)

    np.nan_to_num(feat, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return feat


# ─────────────────────────────────────────────────────────────────────
# SPLIT
# ─────────────────────────────────────────────────────────────────────

def make_splits_from_time_id(df: pd.DataFrame,
                              val_ratio:  float = 0.15,
                              test_ratio: float = 0.15,
                              seed:       int   = 42) -> dict:
    rng    = np.random.default_rng(seed)
    tids   = df["time_id"].unique().copy()
    rng.shuffle(tids)
    n      = len(tids)
    n_test = int(n * test_ratio)
    n_val  = int(n * val_ratio)
    return {"split_summary": {
        "train": {"ids": tids[n_test + n_val:]},
        "val":   {"ids": tids[n_test : n_test + n_val]},
        "test":  {"ids": tids[:n_test]},
    }}


# ─────────────────────────────────────────────────────────────────────
# PROFILING + CLUSTERING
# ─────────────────────────────────────────────────────────────────────

def _vectorised_window_stats(ret_mat: np.ndarray) -> np.ndarray:
    """(T, S) → (S, 5): rv, mean_ret, skew, kurt, max_dd."""
    n_stocks = ret_mat.shape[1]
    out      = np.zeros((n_stocks, 5), dtype=np.float32)
    out[:, 0] = np.sqrt(np.einsum("ij,ij->j", ret_mat, ret_mat))
    out[:, 1] = ret_mat.mean(axis=0)
    std_col   = ret_mat.std(axis=0)
    nontrivial = std_col > 1e-12
    if nontrivial.any():
        out[nontrivial, 2] = skew(ret_mat[:, nontrivial], axis=0, bias=False)
        out[nontrivial, 3] = kurtosis(ret_mat[:, nontrivial], axis=0,
                                      bias=False, fisher=True)
    cum_ret      = np.exp(np.cumsum(ret_mat, axis=0))
    running_max  = np.maximum.accumulate(cum_ret, axis=0)
    out[:, 4]    = ((running_max - cum_ret) / (running_max + EPSILON)).max(axis=0)
    np.nan_to_num(out, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return out


def build_stock_profiles(df: pd.DataFrame,
                          stock_cols: list,
                          train_ids:  np.ndarray) -> pd.DataFrame:
    print("Step 1 — Building stock profiles (train only) …")
    df_s   = df[df["time_id"].isin(train_ids)].sort_values(
        ["time_id", "seconds_in_bucket"]
    )
    # Use first INPUT_LEN rows of each time_id for profiling
    stat_sum, stat_count = (
        np.zeros((len(stock_cols), len(PROFILE_COLS)), dtype=np.float64),
        0,
    )
    for _, grp in df_s.groupby("time_id", sort=False):
        mat = robust_fill(grp[stock_cols].values.astype(np.float32))
        if mat.shape[0] < INPUT_LEN:
            continue
        stat_sum   += _vectorised_window_stats(
            _safe_log_return_matrix(mat[:INPUT_LEN])
        ).astype(np.float64)
        stat_count += 1

    profiles = pd.DataFrame(
        (stat_sum / max(stat_count, 1)).astype(np.float32),
        index=stock_cols, columns=PROFILE_COLS,
    )
    print(f"  Profiles: {profiles.shape}  ({stat_count} train time_ids)")
    return profiles


def find_optimal_k_elbow(profiles: pd.DataFrame,
                          k_range=range(2, 21),
                          random_state: int = 42) -> int:
    X        = StandardScaler().fit_transform(profiles.values)
    inertias = [
        KMeans(k, n_init=10, max_iter=300, random_state=random_state)
        .fit(X).inertia_
        for k in k_range
    ]
    if _KNEED_AVAILABLE:
        kl = KneeLocator(list(k_range), inertias,
                         curve="convex", direction="decreasing")
        return kl.elbow if kl.elbow else 10
    return 10


def cluster_stocks_kmeans(profiles: pd.DataFrame,
                           n_clusters: int = 10,
                           seed:       int = 42) -> pd.Series:
    print(f"\nStep 2 — K-Means (k={n_clusters}) …")
    X      = StandardScaler().fit_transform(profiles.values)
    km     = KMeans(n_clusters=n_clusters, n_init=20, max_iter=500,
                    random_state=seed)
    labels = km.fit_predict(X)
    cm     = pd.Series(labels, index=profiles.index, name="cluster_id")
    print(f"  Cluster sizes: {cm.value_counts().sort_index().to_dict()}")
    return cm


# ─────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────

class ClusterLSTMDataset(Dataset):
    """
    Per time_id yields SAMPLES_PER_TID (3) non-overlapping samples.

    Each 600-row time_id is split into windows of STRIDE=180 steps:
        window i: input  [i*180 : i*180 + 60)
                  target [i*180 + 60 : i*180 + 180)

    Per sample yields:
        x_seq    : (K, INPUT_LEN=60, N_CHANNELS=12)
        x_scalar : (K, n_scalar_per_cluster)
        y        : (K,)   — log(RV_future / RV_input)
        rv_inputs: (K,)   — raw RV_input for retransformation
    """

    def __init__(self, df: pd.DataFrame, stock_cols: list,
                 cluster_map: pd.Series, time_ids: np.ndarray,
                 name: str = ""):
        self.stock_cols  = stock_cols
        cluster_ids      = sorted(cluster_map.unique())
        self.cluster_ids = cluster_ids
        self.n_clusters  = len(cluster_ids)

        self.cluster_col_idx = {
            cid: np.array(
                [stock_cols.index(s)
                 for s in cluster_map.index[cluster_map == cid]],
                dtype=np.int32,
            )
            for cid in cluster_ids
        }

        # Scalar: last / mean / std for each SCALAR_CHANNEL_IDX channel
        self.n_scalar_per_cluster = len(SCALAR_CHANNEL_IDX) * 3   # 15
        self.n_scalar             = self.n_scalar_per_cluster * self.n_clusters

        df     = df[df["time_id"].isin(time_ids)].sort_values(
            ["time_id", "seconds_in_bucket"]
        )

        # Build (time_id, window_offset) index
        self._samples  = []      # list of (tid, offset) tuples
        self._wap_blocks = {}
        for tid, grp in df.groupby("time_id", sort=True):
            mat = robust_fill(grp[stock_cols].values.astype(np.float32))
            if mat.shape[0] < TOTAL_LEN:
                continue
            self._wap_blocks[tid] = mat   # (600, S)
            for w in range(SAMPLES_PER_TID):
                offset = w * STRIDE
                # Ensure enough rows for input + target from this offset
                if offset + INPUT_LEN + TARGET_LEN <= mat.shape[0]:
                    self._samples.append((tid, offset))

        n_tids = len(self._wap_blocks)
        mb = sum(v.nbytes for v in self._wap_blocks.values()) / 1e6
        print(f"  [{name:5s}] {n_tids:>5} time_ids × {SAMPLES_PER_TID} windows "
              f"= {len(self._samples)} samples | cache {mb:.1f} MB")

    def __len__(self) -> int:
        return len(self._samples)

    def __getitem__(self, idx):
        tid, offset = self._samples[idx]
        mat         = self._wap_blocks[tid]           # (600, S)

        # Slice this window
        inp_start = offset
        inp_end   = offset + INPUT_LEN
        tgt_start = inp_end
        tgt_end   = inp_end + TARGET_LEN

        mat_inp = mat[inp_start:inp_end]              # (60, S)
        mat_tgt = mat[tgt_start:tgt_end]              # (120, S)

        open_prices = mat_inp[0]                      # (S,)

        # Feature tensor for the 60-step input window: (60, S, 12)
        feat_all = compute_features_matrix(mat_inp, open_prices)

        ret_inp = _safe_log_return_matrix(mat_inp)
        ret_fut = _safe_log_return_matrix(mat_tgt)

        x_seq    = np.empty((self.n_clusters, INPUT_LEN, N_CHANNELS), dtype=np.float32)
        x_scalar = np.empty((self.n_clusters, self.n_scalar_per_cluster), dtype=np.float32)
        targets  = np.empty(self.n_clusters, dtype=np.float32)
        rv_inputs = np.empty(self.n_clusters, dtype=np.float32)

        for ci, cid in enumerate(self.cluster_ids):
            idx_c = self.cluster_col_idx[cid]

            # Per-cluster mean sequence: (60, 12)
            x_seq[ci] = feat_all[:, idx_c, :].mean(axis=1)

            # Scalar features from HAR-RV channels
            sc = []
            for ch in SCALAR_CHANNEL_IDX:
                ch_data = feat_all[:, idx_c, ch].mean(axis=1)   # (60,)
                sc.extend([float(ch_data[-1]),
                            float(ch_data.mean()),
                            float(ch_data.std())])
            x_scalar[ci] = sc

            r_in          = ret_inp[:, idx_c]
            rv_in         = float(
                np.sqrt(np.einsum("ij,ij->j", r_in, r_in)).mean()
            ) + EPSILON
            rv_inputs[ci] = rv_in

            r_fut         = ret_fut[:, idx_c]
            rv_fut        = float(
                np.sqrt(np.einsum("ij,ij->j", r_fut, r_fut)).mean()
            ) + EPSILON
            targets[ci]   = np.log(rv_fut / rv_in)

        np.nan_to_num(x_seq,    copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.nan_to_num(x_scalar, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

        return (
            torch.from_numpy(x_seq),        # (K, 60, 12)
            torch.from_numpy(x_scalar),     # (K, 15)
            torch.from_numpy(targets),      # (K,)
            torch.from_numpy(rv_inputs),    # (K,)
        )


# ─────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────

class ScalarBranch(nn.Module):
    """MLP over per-cluster HAR-RV scalar features (15 → 32)."""
    def __init__(self, n_scalar_per_cluster: int = 15, hidden: int = 32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_scalar_per_cluster, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.out_features = hidden

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)                     # (N, 32)


class ClusterVolatilityLSTM(nn.Module):
    """
    Multi-stock Cluster LSTM — 60 × 12 architecture.

    Input      → (B, K, 60, 12)
    LSTM × 3   → 256 hidden units per layer, dropout p=0.2 between layers
    FC head    → linear(256 + 32, 1) → scalar per cluster
    Output     → (B, K)
    """
    def __init__(
        self,
        n_clusters:           int   = 10,
        input_dim:            int   = N_CHANNELS,
        hidden_dim:           int   = 256,
        num_layers:           int   = 3,
        dropout:              float = 0.2,
        scalar_hidden:        int   = 32,
        n_scalar_per_cluster: int   = 15,
    ):
        super().__init__()
        self.n_clusters = n_clusters

        # 3-layer LSTM, 256 units, dropout 0.2 between layers
        self.lstm = nn.LSTM(
            input_size  = input_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout,
        )
        self.lstm_drop = nn.Dropout(dropout)

        # Scalar HAR-RV branch
        self.scalar_branch = ScalarBranch(n_scalar_per_cluster, scalar_hidden)

        # Single linear head: concat → scalar
        self.head = nn.Linear(hidden_dim + scalar_hidden, 1)

    def forward(self, x_seq: torch.Tensor,
                x_scalar: torch.Tensor) -> torch.Tensor:
        B, K, T, F = x_seq.shape
        x_flat     = x_seq.reshape(B * K, T, F)
        sc_flat    = x_scalar.reshape(B * K, -1)

        # Final hidden state of the top LSTM layer
        _, (h_n, _) = self.lstm(x_flat)          # h_n: (num_layers, B*K, H)
        z_lstm      = self.lstm_drop(h_n[-1])     # (B*K, 256)

        z_scalar = self.scalar_branch(sc_flat)    # (B*K, 32)
        z        = torch.cat([z_lstm, z_scalar], dim=1)
        out      = self.head(z).squeeze(1)        # (B*K,)
        return out.reshape(B, K)                  # (B, K)


# ─────────────────────────────────────────────────────────────────────
# LOSS
# ─────────────────────────────────────────────────────────────────────

class QLIKELoss(nn.Module):
    """QLIKE in log-ratio space: L = exp(ε) − ε − 1  where ε = y − ŷ."""
    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        eps = y_true - y_pred
        return (torch.exp(eps) - eps - 1.0).mean()


def get_loss_fn(loss_type: str = "qlike") -> nn.Module:
    return QLIKELoss() if loss_type == "qlike" else nn.MSELoss()


# ─────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────

def train_model(model:       nn.Module,
                train_loader: DataLoader,
                val_loader:   DataLoader,
                epochs:       int   = 50,
                lr:           float = 1e-3,
                patience:     int   = 10,
                loss_type:    str   = "qlike",
                ckpt_path:    str   = "best_cluster_lstm.pt") -> nn.Module:

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model   = model.to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=5
    )
    loss_fn = get_loss_fn(loss_type)
    print(f"  Loss: {loss_type.upper()} | Device: {device}")

    best, stale = float("inf"), 0

    for epoch in range(1, epochs + 1):
        t0 = _time.time()

        # ── Train ────────────────────────────────────────────────────
        model.train()
        tr, n = 0.0, 0
        for x_seq, x_sc, yb, _ in train_loader:
            x_seq, x_sc, yb = x_seq.to(device), x_sc.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x_seq, x_sc), yb)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tr += loss.item() * x_seq.size(0)
            n  += x_seq.size(0)
        tr /= max(n, 1)

        # ── Validate ─────────────────────────────────────────────────
        model.eval()
        vl, n = 0.0, 0
        with torch.no_grad():
            for x_seq, x_sc, yb, _ in val_loader:
                x_seq, x_sc, yb = (
                    x_seq.to(device), x_sc.to(device), yb.to(device)
                )
                vl += loss_fn(model(x_seq, x_sc), yb).item() * x_seq.size(0)
                n  += x_seq.size(0)
        vl /= max(n, 1)

        sched.step(vl)
        improved = vl < best
        tag = ""
        if improved:
            best, stale = vl, 0
            torch.save(model.state_dict(), ckpt_path)
            tag = " ★"
        else:
            stale += 1

        if epoch == 1 or epoch % 5 == 0 or improved:
            print(f"  Epoch {epoch:>3}/{epochs} | Train {tr:.6f} | "
                  f"Val {vl:.6f} | {_time.time()-t0:.1f}s{tag}")

        if stale >= patience:
            print(f"  Early stop at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    return model


# ─────────────────────────────────────────────────────────────────────
# SMEARING  (per-cluster Duan correction)
# ─────────────────────────────────────────────────────────────────────

def compute_smearing_factor(model: nn.Module,
                             loader: DataLoader) -> np.ndarray:
    """
    Per-cluster Duan smearing:  δ̂_k = E[exp(y_k − ŷ_k)]

    Corrects Jensen's inequality bias when exponentiating log-ratio
    predictions back to RV space.
    """
    device = next(model.parameters()).device
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for x_seq, x_sc, yb, _ in loader:
            preds.append(
                model(x_seq.to(device), x_sc.to(device)).cpu().numpy()
            )
            actuals.append(yb.numpy())
    preds   = np.concatenate(preds)     # (N, K)
    actuals = np.concatenate(actuals)   # (N, K)
    delta   = np.mean(np.exp(actuals - preds), axis=0)   # (K,)
    print(f"  Per-cluster smearing δ̂: {np.round(delta, 4).tolist()}")
    return delta


# ─────────────────────────────────────────────────────────────────────
# EVALUATION — dual output: with and without smearing
# ─────────────────────────────────────────────────────────────────────

def _rv_metrics(rv_pred: np.ndarray, rv_act: np.ndarray) -> dict:
    """Compute the standard suite of volatility forecast metrics."""
    fp  = np.maximum(rv_pred.ravel(), EPSILON)
    fa  = np.maximum(rv_act.ravel(),  EPSILON)
    mse = mean_squared_error(fa, fp)
    pct = (fp - fa) / fa
    ratio = fa / (fp + EPSILON)
    return dict(
        mse   = float(mse),
        rmse  = float(np.sqrt(mse)),
        r2    = float(r2_score(fa, fp)),
        mae   = float(np.mean(np.abs(fp - fa))),
        rmspe = float(np.sqrt(np.mean(pct ** 2))),
        mape  = float(np.mean(np.abs(pct))),
        qlike = float(np.mean(ratio - np.log(ratio + EPSILON) - 1)),
    )


def _print_metrics(m: dict, label: str) -> None:
    w = 48
    print(f"\n  ┌{'─' * w}┐")
    print(f"  │  {label:<{w - 2}}│")
    print(f"  ├{'─' * w}┤")
    print(f"  │  MSE   : {m['mse']:<12.8f}   RMSE  : {m['rmse']:<12.8f}│")
    print(f"  │  R²    : {m['r2']:<12.6f}   MAE   : {m['mae']:<12.8f}│")
    print(f"  │  RMSPE : {m['rmspe']:<12.6f}   MAPE  : {m['mape']:<12.6f}│")
    print(f"  │  QLIKE : {m['qlike']:<12.6f}{'':>22}│")
    print(f"  └{'─' * w}┘")


def evaluate_model(model:    nn.Module,
                   loader:   DataLoader,
                   smearing: np.ndarray,
                   label:    str = "") -> dict:
    """
    Retransform log-ratio predictions → RV space and report metrics
    for BOTH the raw (no-smearing) and smearing-corrected forecasts.
    """
    device = next(model.parameters()).device
    model.eval()
    logp_list, logy_list, rvin_list = [], [], []

    with torch.no_grad():
        for x_seq, x_sc, yb, rv_in in loader:
            logp_list.append(
                model(x_seq.to(device), x_sc.to(device)).cpu().numpy()
            )
            logy_list.append(yb.numpy())
            rvin_list.append(rv_in.numpy())

    logp = np.concatenate(logp_list)    # (N, K)
    logy = np.concatenate(logy_list)    # (N, K)
    rvin = np.concatenate(rvin_list)    # (N, K)

    rv_act = np.exp(logy) * rvin

    # ── Without smearing ─────────────────────────────────────────────
    rv_raw = np.exp(logp) * rvin
    m_raw  = _rv_metrics(rv_raw, rv_act)

    # ── With per-cluster Duan smearing ───────────────────────────────
    rv_sme = np.exp(logp) * smearing[None, :] * rvin
    m_sme  = _rv_metrics(rv_sme, rv_act)

    if label:
        print(f"\n{'═' * 52}")
        print(f"  {label}")
        print(f"{'═' * 52}")

    _print_metrics(m_raw, "No Smearing   — exp(ŷ) × RV_input")
    _print_metrics(m_sme, "Smearing-Corrected — exp(ŷ) × δ̂ × RV_input")

    return {"no_smearing": m_raw, "smearing": m_sme}


# ─────────────────────────────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_lstm_cluster_pipeline(
    df:              pd.DataFrame,
    stock_cols:      list,
    splits:          dict,
    n_clusters:      int   = None,
    k_range               = range(2, 21),
    seed:            int   = 42,
    hidden_dim:      int   = 256,
    num_layers:      int   = 3,
    scalar_hidden:   int   = 32,
    dropout:         float = 0.2,
    epochs:          int   = 50,
    bs:              int   = 32,
    lr:              float = 1e-3,
    patience:        int   = 10,
    loss_type:       str   = "qlike",
    workers:         int   = 0,
    pretrained_path: str   = None,
    ckpt_path:       str   = "best_cluster_lstm.pt",
) -> dict:

    print("=" * 60)
    print("Multi-Stock Cluster LSTM Volatility Pipeline")
    print(f"  Windowing : {INPUT_LEN} input + {TARGET_LEN} target = "
          f"{STRIDE} stride, {SAMPLES_PER_TID} samples/time_id")
    print(f"  Input shape : (K, {INPUT_LEN}, {N_CHANNELS})")
    print(f"  LSTM        : {num_layers} layers × {hidden_dim} units, "
          f"dropout={dropout}")
    print("=" * 60)

    train_ids = splits["split_summary"]["train"]["ids"]
    val_ids   = splits["split_summary"]["val"]["ids"]
    test_ids  = splits["split_summary"]["test"]["ids"]
    print(f"Split: train={len(train_ids)} | val={len(val_ids)} | "
          f"test={len(test_ids)}")

    # ── Step 1 & 2: Profile + Cluster ────────────────────────────────
    profiles = build_stock_profiles(df, stock_cols, train_ids)
    if n_clusters is None:
        n_clusters = find_optimal_k_elbow(
            profiles, k_range=k_range, random_state=seed
        )
        print(f"  Elbow → k={n_clusters}")
    cluster_map = cluster_stocks_kmeans(profiles, n_clusters, seed)

    # ── Step 3: Datasets ─────────────────────────────────────────────
    print(f"\nBuilding datasets ({INPUT_LEN}-step input, "
          f"{TARGET_LEN}-step target, {N_CHANNELS} channels) …")
    train_ds = ClusterLSTMDataset(df, stock_cols, cluster_map, train_ids, "train")
    val_ds   = ClusterLSTMDataset(df, stock_cols, cluster_map, val_ids,   "val")
    test_ds  = ClusterLSTMDataset(df, stock_cols, cluster_map, test_ids,  "test")

    pin = torch.cuda.is_available()
    tl  = DataLoader(train_ds, batch_size=bs, shuffle=True,
                     num_workers=workers, pin_memory=pin)
    vl  = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                     num_workers=workers, pin_memory=pin)
    sl  = DataLoader(test_ds,  batch_size=bs, shuffle=False,
                     num_workers=workers, pin_memory=pin)

    # Sanity check
    x_seq, x_sc, yb, rv_in = next(iter(tl))
    print(f"\nSanity: seq {tuple(x_seq.shape)} | scalar {tuple(x_sc.shape)} | "
          f"y {tuple(yb.shape)} | rv_in {tuple(rv_in.shape)}")
    assert x_seq.shape[2] == INPUT_LEN, \
        f"Expected T={INPUT_LEN}, got {x_seq.shape[2]}"
    assert x_seq.shape[3] == N_CHANNELS, \
        f"Expected F={N_CHANNELS}, got {x_seq.shape[3]}"
    assert torch.isfinite(x_seq).all() and torch.isfinite(yb).all(), \
        "NaN/Inf detected in first batch!"

    # ── Step 4: Model ────────────────────────────────────────────────
    n_scalar_pc = train_ds.n_scalar_per_cluster
    model = ClusterVolatilityLSTM(
        n_clusters           = n_clusters,
        input_dim            = N_CHANNELS,       # 12
        hidden_dim           = hidden_dim,        # 256
        num_layers           = num_layers,        # 3
        dropout              = dropout,           # 0.2
        scalar_hidden        = scalar_hidden,
        n_scalar_per_cluster = n_scalar_pc,
    )

    if pretrained_path:
        model.load_state_dict(
            torch.load(pretrained_path, map_location="cpu")
        )
        print(f"  Loaded weights from {pretrained_path}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel: ClusterVolatilityLSTM")
    print(f"  Input  : (K={n_clusters}, T={INPUT_LEN}, F={N_CHANNELS})")
    print(f"  LSTM   : {num_layers} layers × {hidden_dim} hidden, dropout={dropout}")
    print(f"  Scalar : {n_scalar_pc} per cluster → MLP({scalar_hidden})")
    print(f"  Head   : Linear({hidden_dim + scalar_hidden}, 1)")
    print(f"  Params : {n_params:,}\n")

    # ── Step 5: Train ────────────────────────────────────────────────
    print("Training …")
    model = train_model(
        model, tl, vl,
        epochs=epochs, lr=lr, patience=patience,
        loss_type=loss_type, ckpt_path=ckpt_path,
    )

    # ── Step 6: Smearing  (estimated on train set) ───────────────────
    print("\nSmearing factors (estimated on train set):")
    delta = compute_smearing_factor(model, tl)

    # ── Step 7: Evaluate — both with and without smearing ────────────
    val_metrics  = evaluate_model(model, vl, delta,
                                  "Validation — smearing vs raw")
    test_metrics = evaluate_model(model, sl, delta,
                                  "TEST SET   — smearing vs raw  "
                                  "[touched once]")

    return {
        "model":        model,
        "profiles":     profiles,
        "cluster_map":  cluster_map,
        "smearing":     delta,
        "val_metrics":  val_metrics,
        "test_metrics": test_metrics,
        "n_clusters":   n_clusters,
    }


# ─────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────

def main(
    df:          pd.DataFrame,
    n_clusters:  int   = None,
    epochs:      int   = 100,
    bs:          int   = 32,
    lr:          float = 1e-3,
    patience:    int   = 10,
    loss_type:   str   = "qlike",
    dropout:     float = 0.2,
    hidden_dim:  int   = 256,
    num_layers:  int   = 3,
    seed:        int   = 42,
    val_ratio:   float = 0.15,
    test_ratio:  float = 0.15,
    workers:     int   = 0,
    ckpt_path:   str   = "best_cluster_lstm.pt",
) -> dict:

    print("Loading data …")
    stock_cols = [
        c for c in df.columns
        if c not in ("time_id", "seconds_in_bucket")
    ]
    print(f"  shape: {df.shape} | stocks: {len(stock_cols)} | "
          f"tids: {df['time_id'].nunique()}")

    splits = make_splits_from_time_id(df, val_ratio, test_ratio, seed)

    results = run_lstm_cluster_pipeline(
        df, stock_cols, splits,
        n_clusters  = n_clusters,
        seed        = seed,
        hidden_dim  = hidden_dim,
        num_layers  = num_layers,
        dropout     = dropout,
        epochs      = epochs,
        bs          = bs,
        lr          = lr,
        patience    = patience,
        loss_type   = loss_type,
        workers     = workers,
        ckpt_path   = ckpt_path,
    )

    # ── Final summary ─────────────────────────────────────────────────
    tm_raw = results["test_metrics"]["no_smearing"]
    tm_sme = results["test_metrics"]["smearing"]

    print("\n" + "=" * 60)
    print("Pipeline complete — Test Set Summary")
    print("=" * 60)
    print(f"  {'Metric':<10}  {'No Smearing':>14}  {'Smearing':>14}")
    print(f"  {'-'*10}  {'-'*14}  {'-'*14}")
    for key in ("qlike", "rmse", "rmspe", "mape", "r2"):
        print(f"  {key.upper():<10}  "
              f"{tm_raw[key]:>14.6f}  {tm_sme[key]:>14.6f}")
    print("=" * 60)

    return results


if __name__ == "__main__":
    # df: wide DataFrame with columns
    #     time_id | seconds_in_bucket | stock_1 | stock_2 | … | stock_N
    main(
        df,
        n_clusters = None,      # None → auto-detect via elbow
        loss_type  = "mse",
        epochs     = 100,
        bs         = 32,
        hidden_dim = 256,
        num_layers = 3,
        dropout    = 0.2,
    )

Loading data …
  shape: (2298000, 114) | stocks: 112 | tids: 3830
Multi-Stock Cluster LSTM Volatility Pipeline
  Input shape : (K, 60, 12)
  LSTM        : 3 layers × 256 units, dropout=0.2
Split: train=2682 | val=574 | test=574
Step 1 — Building stock profiles (train only) …
  Profiles: (112, 5)  (2682 train time_ids)
  Elbow → k=10

Step 2 — K-Means (k=10) …
  Cluster sizes: {0: 8, 1: 15, 2: 17, 3: 7, 4: 19, 5: 13, 6: 4, 7: 14, 8: 4, 9: 11}

Building datasets (60-step windows, 12 channels) …
  [train]  2682 time_ids | cache 720.9 MB
  [val  ]   574 time_ids | cache 154.3 MB
  [test ]   574 time_ids | cache 154.3 MB

Sanity: seq (32, 10, 60, 12) | scalar (32, 10, 15) | y (32, 10) | rv_in (32, 10)

Model: ClusterVolatilityLSTM
  Input  : (K=10, T=60, F=12)
  LSTM   : 3 layers × 256 hidden, dropout=0.2
  Scalar : 15 per cluster → MLP(32)
  Head   : Linear(288, 1)
  Params : 1,331,009

Training …
  Loss: MSE | Device: cpu
  Epoch   1/100 | Train 0.029633 | Val 0.026179 | 437.4s ★
  Epoch 